In [1]:
import json
import re
import pandas as pd
import numpy as np
JSON_PATH = "afl0-1.json"

with open(JSON_PATH) as f:
    data = json.load(f)

def as_int(x):
    try:
        return int(x)
    except Exception:
        return None

def iter_items(x):
    """
    Support both list- and dict-based antiSMASH JSON structures
    """
    if x is None:
        return []
    if isinstance(x, list):
        return [v for v in x if isinstance(v, dict)]
    if isinstance(x, dict):
        return [v for v in x.values() if isinstance(v, dict)]
    return []

rows = []

records = data.get("records", [])
print("records:", len(records))

for rec_i, rec in enumerate(records):
    if not isinstance(rec, dict):
        continue

    seqname = rec.get("id") or rec.get("name") or rec.get("record_id") or f"record_{rec_i+1}"
    areas = rec.get("areas", [])
    if not isinstance(areas, list):
        continue

    for a_i, area in enumerate(areas):
        if not isinstance(area, dict):
            continue

        # Convert the 0-based antiSMASH start coordinate to 1-based coordinates
        area_start = as_int(area.get("start"))
        area_end   = as_int(area.get("end"))

        if area_start is not None:
            area_start += 1

        products = area.get("products")
        if isinstance(products, list):
            products_str = ",".join(map(str, products))
        else:
            products_str = str(products) if products is not None else None

        # Store the complete area when valid coordinates are available
        if area_start is not None and area_end is not None and area_end >= area_start:
            rows.append({
                "level": "area",
                "cluster_id": f"{seqname}:area{a_i+1}",   # 1-based numbering
                "seqname": seqname,
                "start": area_start,
                "end": area_end,
                "bgc_type": products_str,
                "product": products_str,
                "kind": None,
                "category": None,
                "area_number": a_i + 1,
                "candidate_number": None
            })

        # Extract candidate regions
        candidates = iter_items(area.get("candidates"))
        for c_i, c in enumerate(candidates):
            c_start = as_int(c.get("start"))
            c_end   = as_int(c.get("end"))

            # Convert the start coordinate to 1-based coordinates
            if c_start is not None:
                c_start += 1

            if c_start is None or c_end is None or c_end < c_start:
                continue

            candidate_number = c.get("candidate_cluster_number")
            if candidate_number is None:
                candidate_number = c_i + 1   # Assign 1-based numbering when absent

            rows.append({
                "level": "candidate",
                "cluster_id": f"{seqname}:area{a_i+1}:cand{candidate_number}",
                "seqname": seqname,
                "start": c_start,
                "end": c_end,
                "bgc_type": c.get("kind"),
                "product": products_str,
                "kind": c.get("kind"),
                "category": None,
                "area_number": a_i + 1,
                "candidate_number": candidate_number
            })

bgc = pd.DataFrame(rows)

print("Extracted rows:", len(bgc))
print("Counts by level:")
print(bgc["level"].value_counts())

# Remove duplicates
bgc = (
    bgc.dropna(subset=["seqname", "start", "end"])
       .astype({"start": int, "end": int})
       .sort_values(["seqname", "start", "end", "level"])
       .drop_duplicates(subset=["seqname", "start", "end", "level", "cluster_id"])
       .reset_index(drop=True)
)

# Save outputs
bgc.to_csv("afl0-1_bgc_coordinates_area_candidate_fixed.tsv", sep="\t", index=False)
bgc[bgc["level"] == "candidate"].to_csv("afl0-1_bgc_coordinates_candidate_fixed.tsv", sep="\t", index=False)
bgc[bgc["level"] == "area"].to_csv("afl0-1_bgc_coordinates_area_fixed.tsv", sep="\t", index=False)

print("Saved:")
print("  afl0-1_bgc_coordinates_area_candidate_fixed.tsv")
print("  afl0-1_bgc_coordinates_candidate_fixed.tsv")
print("  afl0-1_bgc_coordinates_area_fixed.tsv")

display(bgc.head(10))

records: 138
Extracted rows: 203
Counts by level:
level
candidate    119
area          84
Name: count, dtype: int64
Saved:
  afl0-1_bgc_coordinates_area_candidate_fixed.tsv
  afl0-1_bgc_coordinates_candidate_fixed.tsv
  afl0-1_bgc_coordinates_area_fixed.tsv


,level,cluster_id,seqname,start,end,bgc_type,product,kind,category,area_number,candidate_number
0,area,1041045516891:area1,1041045516891,1,64997,NRPS,NRPS,None,None,1,NaN
1,candidate,1041045516891:area1:cand1,1041045516891,1,64997,single,NRPS,single,None,1,1.0
2,area,1041045516891:area2,1041045516891,71963,136745,NRPS,NRPS,None,None,2,NaN
3,candidate,1041045516891:area2:cand1,1041045516891,71963,136745,single,NRPS,single,None,2,1.0
4,candidate,1041045516891:area3:cand1,1041045516891,616217,646846,single,"terpene,NRPS,terpene-precursor,T1PKS",single,None,3,1.0
5,area,1041045516891:area3,1041045516891,616217,783085,"terpene,NRPS,terpene-precursor,T1PKS","terpene,NRPS,terpene-precursor,T1PKS",None,None,3,NaN
6,candidate,1041045516891:area3:cand3,1041045516891,625460,701802,single,"terpene,NRPS,terpene-precursor,T1PKS",single,None,3,3.0
7,candidate,1041045516891:area3:cand2,1041045516891,625460,783085,neighbouring,"terpene,NRPS,terpene-precursor,T1PKS",neighbouring,None,3,2.0
8,candidate,1041045516891:area3:cand4,1041045516891,673685,708451,single,"terpene,NRPS,terpene-precursor,T1PKS",single,None,3,4.0
9,candidate,1041045516891:area3:cand5,1041045516891,674693,746533,chemical_hybrid,"terpene,NRPS,terpene-precursor,T1PKS",chemical_hybrid,None,3,5.0


In [2]:

GTF_PATH = "afl0-1.gtf"      # Input filename
ZN_LIST  = "afl0-1_48_gene_ortholog"      # Ortholog ID list

gtf_cols = ["seqname","source","feature","start","end","score","strand","frame","attribute"]
gtf = pd.read_csv(GTF_PATH, sep="\t", comment="#", names=gtf_cols)

def extract_gene_id(attr):
    m = re.search(r'gene_id "([^"]+)"', str(attr))
    if m: return m.group(1)
    m = re.search(r'ID=([^;]+)', str(attr))
    if m: return m.group(1)
    return None

gtf["gene_id"] = gtf["attribute"].apply(extract_gene_id)

# Treat gene IDs with at least one CDS feature as protein-coding genes.
# Aggregate all features by gene ID to support GTF files without gene features.
cds_gene_ids = set(gtf.loc[gtf["feature"] == "CDS", "gene_id"].dropna())

def unique_or_first(values):
    vals = [v for v in pd.Series(values).dropna().unique() if v != "."]
    return vals[0] if vals else None

genes = (gtf[gtf["gene_id"].isin(cds_gene_ids)]
           .groupby(["seqname", "gene_id"], as_index=False)
           .agg(start=("start", "min"),
                end=("end", "max"),
                strand=("strand", unique_or_first)))

zn_ids = set(x.strip() for x in open(ZN_LIST) if x.strip() and not x.startswith("#"))
zn = genes[genes["gene_id"].isin(zn_ids)].copy()

missing_zn_ids = sorted(zn_ids - set(zn["gene_id"]))

print("Total protein-coding genes:", len(genes))
print("Zn2Cys6 IDs in list:", len(zn_ids))
print("Zn2Cys6 genes matched in GTF:", len(zn))
print("Missing Zn2Cys6 IDs:", len(missing_zn_ids))
if missing_zn_ids:
    print(missing_zn_ids[:20])


Total protein-coding genes: 12604
Zn2Cys6 IDs in list: 28
Zn2Cys6 genes matched in GTF: 28
Missing Zn2Cys6 IDs: 0


In [3]:

bgc = pd.read_csv("afl0-1_bgc_coordinates_area_fixed.tsv", sep="\t").copy()

FLANK = 20000  # 20 kb

zn["mid"] = ((zn["start"] + zn["end"]) // 2).astype(int)
zn["within_bgc"] = False
zn["near_bgc"] = False
zn["nearest_dist"] = np.nan
zn["nearest_bgc_id"] = None
zn["nearest_bgc_seqname"] = None
zn["nearest_bgc_start"] = np.nan
zn["nearest_bgc_end"] = np.nan
zn["nearest_bgc_type"] = None
zn["nearest_bgc_product"] = None
zn["nearest_bgc_area_number"] = np.nan
zn["gene_bgc_relation"] = None
zn["signed_dist_to_bgc"] = np.nan

def interval_distance(gene_start, gene_end, bgc_start, bgc_end):
    """Return the minimum distance between gene and BGC intervals; return 0 for overlap."""
    if gene_end < bgc_start:
        return bgc_start - gene_end
    if gene_start > bgc_end:
        return gene_start - bgc_end
    return 0

def relation_to_bgc(gene_start, gene_end, bgc_start, bgc_end):
    if gene_end < bgc_start:
        return "left_of_bgc"
    if gene_start > bgc_end:
        return "right_of_bgc"
    return "overlaps_bgc"

def signed_distance_to_bgc(gene_start, gene_end, bgc_start, bgc_end):
    """Return a negative value to the left of the BGC, a positive value to the right, and 0 for overlap."""
    if gene_end < bgc_start:
        return -(bgc_start - gene_end)
    if gene_start > bgc_end:
        return gene_start - bgc_end
    return 0

for chr_name, sub in zn.groupby("seqname"):
    bgc_chr = bgc[bgc["seqname"] == chr_name]
    if bgc_chr.empty:
        continue

    for idx, row in sub.iterrows():
        best = None

        for _, b in bgc_chr.iterrows():
            d = interval_distance(row["start"], row["end"], b["start"], b["end"])
            if best is None or d < best["dist"]:
                best = {
                    "dist": d,
                    "bgc": b,
                    "relation": relation_to_bgc(row["start"], row["end"], b["start"], b["end"]),
                    "signed_dist": signed_distance_to_bgc(row["start"], row["end"], b["start"], b["end"]),
                }

        if best is None:
            continue

        b = best["bgc"]
        zn.loc[idx, "nearest_dist"] = best["dist"]
        zn.loc[idx, "nearest_bgc_id"] = b["cluster_id"]
        zn.loc[idx, "nearest_bgc_seqname"] = b["seqname"]
        zn.loc[idx, "nearest_bgc_start"] = b["start"]
        zn.loc[idx, "nearest_bgc_end"] = b["end"]
        zn.loc[idx, "nearest_bgc_type"] = b.get("bgc_type")
        zn.loc[idx, "nearest_bgc_product"] = b.get("product")
        zn.loc[idx, "nearest_bgc_area_number"] = b.get("area_number")
        zn.loc[idx, "gene_bgc_relation"] = best["relation"]
        zn.loc[idx, "signed_dist_to_bgc"] = best["signed_dist"]
        zn.loc[idx, "within_bgc"] = best["dist"] == 0
        zn.loc[idx, "near_bgc"] = best["dist"] <= FLANK

zn_out_cols = [
    "gene_id", "seqname", "start", "end", "strand", "mid",
    "within_bgc", "near_bgc", "nearest_dist", "signed_dist_to_bgc", "gene_bgc_relation",
    "nearest_bgc_id", "nearest_bgc_seqname", "nearest_bgc_start", "nearest_bgc_end",
    "nearest_bgc_type", "nearest_bgc_product", "nearest_bgc_area_number",
]
zn_all = zn[zn_out_cols].sort_values(["seqname", "start", "gene_id"]).reset_index(drop=True)


reference_set = zn_all[zn_all["near_bgc"]].copy().reset_index(drop=True)
reference_set.insert(0, "reference_species", "Aspergillus_flavus_NRRL3357")
reference_set.insert(1, "reference_set", f"Zn2Cys6_near_area_BGC_{FLANK//1000}kb")
reference_set["flank_bp"] = FLANK

print("Within area BGC:", int(zn_all["within_bgc"].sum()))
print(f"Near area BGC ({FLANK//1000} kb):", int(zn_all["near_bgc"].sum()))

zn_all.to_csv("afl0-1_zn_bgc_proximity_area_fixed.tsv", sep="\t", index=False)
reference_set.to_csv("aflavus_bgc_reference_set_20kb.tsv", sep="\t", index=False)

print("Saved:")
print("  afl0-1_zn_bgc_proximity_area_fixed.tsv")
print("  aflavus_bgc_reference_set_20kb.tsv")

display(reference_set.head())


Within area BGC: 20
Near area BGC (20 kb): 27
Saved:
  afl0-1_zn_bgc_proximity_area_fixed.tsv
  aflavus_bgc_reference_set_20kb.tsv


,reference_species,reference_set,gene_id,seqname,start,end,strand,mid,within_bgc,near_bgc,...,signed_dist_to_bgc,gene_bgc_relation,nearest_bgc_id,nearest_bgc_seqname,nearest_bgc_start,nearest_bgc_end,nearest_bgc_type,nearest_bgc_product,nearest_bgc_area_number,flank_bp
0,Aspergillus_flavus_NRRL3357,Zn2Cys6_near_area_BGC_20kb,AFL2G_07274,1041045516891,65641,68076,+,66858,False,True,...,644.0,right_of_bgc,1041045516891:area1,1041045516891,1.0,64997.0,NRPS,NRPS,1.0,20000
1,Aspergillus_flavus_NRRL3357,Zn2Cys6_near_area_BGC_20kb,AFL2G_07508,1041045516891,717626,719901,-,718763,True,True,...,0.0,overlaps_bgc,1041045516891:area3,1041045516891,616217.0,783085.0,"terpene,NRPS,terpene-precursor,T1PKS","terpene,NRPS,terpene-precursor,T1PKS",3.0,20000
2,Aspergillus_flavus_NRRL3357,Zn2Cys6_near_area_BGC_20kb,AFL2G_07589,1041045516891,936215,938883,-,937549,False,True,...,-17432.0,left_of_bgc,1041045516891:area4,1041045516891,956315.0,987150.0,terpene,terpene,4.0,20000
3,Aspergillus_flavus_NRRL3357,Zn2Cys6_near_area_BGC_20kb,AFL2G_07831,1041045516891,1666600,1668108,-,1667354,True,True,...,0.0,overlaps_bgc,1041045516891:area6,1041045516891,1653701.0,1686318.0,terpene,terpene,6.0,20000
4,Aspergillus_flavus_NRRL3357,Zn2Cys6_near_area_BGC_20kb,AFL2G_07887,1041045516891,1861877,1864430,-,1863153,True,True,...,0.0,overlaps_bgc,1041045516891:area7,1041045516891,1789006.0,1875781.0,"NRPS,NRPS-like","NRPS,NRPS-like",7.0,20000
